# Per-Call Metadata, Operation Chains & Revisions

This notebook demonstrates three pipeline features using the `attested_openai` wrapper:

1. **Per-call metadata (`glacis_tags`)** — Tag individual LLM calls with document name, chunk ID, step name, etc.
2. **Operation chains (`glacis_operation`)** — Group related calls under a shared `operation_id` with auto-incrementing sequence.
3. **Revision chains (`op.supersedes`)** — Link a corrected attestation to the one it replaces.

**What we'll build:** A 3-step content pipeline:
1. **Generate** a draft paragraph (sequence 0, tagged `step: generate`)
2. **Review** the draft for quality issues (sequence 1, tagged `step: review`)
3. **Revise** the draft based on feedback (sequence 2, tagged `step: revise`, supersedes the original)

```bash
pip install glacis[openai]
```

Set `OPENAI_API_KEY` in your environment or in `tests/.env`.

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Load .env
env_path = REPO_ROOT / "tests" / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
SIGNING_SEED = b"demo-chain-revision-seed00000000"  # 32 bytes

print("Setup complete")

## Key Concepts

### Per-call metadata (`glacis_tags`)

The `metadata=` parameter on `attested_openai()` sets **static** tags on every call. Use `glacis_tags()` to set tags for individual calls:

```python
# Static metadata (every call)
client = attested_openai(metadata={"pipeline": "qa-gen"})

# Per-call metadata (this call only)
with client.glacis_tags({"document": "Eliquis_PI.pdf", "chunk_id": "c012"}):
    response = client.chat.completions.create(...)
```

Per-call tags merge with and override static tags. The reserved keys `provider` and `model` cannot be overridden.

### Operation chain (`glacis_operation`)

Groups related calls under a shared `operation_id` with auto-incrementing `operation_sequence`:

```
operation_id: "abc-123"
┌──────────┐    ┌──────────┐    ┌──────────┐
│ Generate │───>│ Review   │───>│ Revise   │
│ seq: 0   │    │ seq: 1   │    │ seq: 2   │
└──────────┘    └──────────┘    └──────────┘
```

### Revision (`op.supersedes`)

Links a new attestation to the one it replaces. One-shot — applies only to the next call.

```
Generate (v1) ─── superseded by ───> Revise (v2)
```

## Initialize the Attested Client

We use `attested_openai` in offline mode. The `metadata=` parameter sets static tags that appear on every attestation.

In [ ]:
from glacis.integrations.openai import attested_openai, get_last_receipt
from glacis.integrations.openai import get_evidence

client = attested_openai(
    openai_api_key=OPENAI_API_KEY,
    offline=True,
    signing_seed=SIGNING_SEED,
    metadata={"pipeline": "content-review", "team": "devrel"},  # on every call
)

print("Attested OpenAI client ready (offline mode)")
print("Static metadata: pipeline=content-review, team=devrel")

## Step 1: Generate a Draft (sequence 0)

We use both `glacis_operation()` (to start a chain) and `glacis_tags()` (to tag this call as the `generate` step).

In [ ]:
draft_prompt = (
    "Write a single concise paragraph (3-4 sentences) about the importance "
    "of API rate limiting for a developer blog. Be specific and technical."
)

with client.glacis_operation() as op:

    # --- Step 0: Generate ---
    with client.glacis_tags({"step": "generate", "topic": "rate-limiting"}):
        draft_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": draft_prompt}],
            temperature=0.7,
        )
        draft_receipt = get_last_receipt()

    draft_text = draft_response.choices[0].message.content
    print("Draft:")
    print(draft_text)
    print(f"\nAttestation: {draft_receipt.id}")
    print(f"Operation:   {draft_receipt.operation_id}")
    print(f"Sequence:    {draft_receipt.operation_sequence}")

    # --- Step 1: Review ---
    review_prompt = (
        f"Review the following paragraph for a developer blog. "
        f"Identify 2-3 specific issues (clarity, accuracy, tone, or missing details). "
        f"Be concise \u2014 one sentence per issue.\n\n"
        f"Paragraph:\n{draft_text}"
    )

    with client.glacis_tags({"step": "review", "topic": "rate-limiting"}):
        review_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": review_prompt}],
            temperature=0.3,
        )
        review_receipt = get_last_receipt()

    review_text = review_response.choices[0].message.content
    print(f"\n{'='*60}")
    print("Review Feedback:")
    print(review_text)
    print(f"\nAttestation: {review_receipt.id}")
    print(f"Sequence:    {review_receipt.operation_sequence}")

    # --- Step 2: Revise (supersedes the draft) ---
    revise_prompt = (
        f"Revise the following paragraph based on the review feedback below. "
        f"Keep it as a single paragraph (3-4 sentences), concise and technical.\n\n"
        f"Original paragraph:\n{draft_text}\n\n"
        f"Review feedback:\n{review_text}"
    )

    op.supersedes(draft_receipt.id)  # next call replaces the draft

    with client.glacis_tags({"step": "revise", "topic": "rate-limiting"}):
        revise_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": revise_prompt}],
            temperature=0.5,
        )
        revise_receipt = get_last_receipt()

    revised_text = revise_response.choices[0].message.content
    print(f"\n{'='*60}")
    print("Revised Draft:")
    print(revised_text)
    print(f"\nAttestation: {revise_receipt.id}")
    print(f"Sequence:    {revise_receipt.operation_sequence}")
    print(f"Supersedes:  {revise_receipt.supersedes}  \u2190 original draft")

## Full Audit Trail

All three attestations share the same `operation_id`. Each has step-specific metadata from `glacis_tags()` merged with the static pipeline/team metadata.

In [ ]:
all_receipts = [
    ("Generate (v1)", draft_receipt),
    ("Review",        review_receipt),
    ("Revise (v2)",   revise_receipt),
]

print("=" * 60)
print("AUDIT TRAIL")
print("=" * 60)
print(f"\nOperation ID: {op.operation_id}\n")

for label, receipt in all_receipts:
    supersedes_info = ""
    if receipt.supersedes:
        for other_label, other in all_receipts:
            if other.id == receipt.supersedes:
                supersedes_info = f"  \u2190 supersedes {other_label}"
                break

    print(f"  [{receipt.operation_sequence}] {label}{supersedes_info}")
    print(f"      ID:   {receipt.id}")
    print(f"      Hash: {receipt.evidence_hash[:40]}...")
    if receipt.supersedes:
        print(f"      Supersedes: {receipt.supersedes}")
    print()

print("Chain Visualization:")
print()
print("  Sequence:")
print("    Generate \u2500\u2500[0]\u2500\u2500> Review \u2500\u2500[1]\u2500\u2500> Revise \u2500\u2500[2]")
print()
print("  Revision:")
print("    Generate (v1) \u2500\u2500\u2500 superseded by \u2500\u2500\u2500> Revise (v2)")

## Inspecting Per-Call Metadata in Evidence

The metadata from `glacis_tags()` is stored in the local evidence store alongside the input, output, and control plane results. Let's verify each step got its own tags.

In [ ]:
for label, receipt in all_receipts:
    evidence = get_evidence(receipt.id)
    md = evidence.get("metadata", {}) if evidence else {}
    print(f"[{receipt.operation_sequence}] {label}")
    print(f"    metadata: {md}")
    print()

In [ ]:
# Verify chain integrity
assert draft_receipt.operation_id == review_receipt.operation_id == revise_receipt.operation_id
assert draft_receipt.operation_sequence == 0
assert review_receipt.operation_sequence == 1
assert revise_receipt.operation_sequence == 2
assert revise_receipt.supersedes == draft_receipt.id
assert draft_receipt.supersedes is None
assert review_receipt.supersedes is None

print("Chain integrity verified:")
print("  \u2713 All 3 attestations share the same operation_id")
print("  \u2713 Sequences are ordered: 0 \u2192 1 \u2192 2")
print("  \u2713 Revised draft supersedes the original")
print("  \u2713 Per-call metadata is step-specific")
print("  \u2713 Static metadata (pipeline, team) present on every call")

## Summary

### Three Pipeline Features

| Feature | API | Purpose |
|---------|-----|--------|
| Per-call metadata | `client.glacis_tags({...})` | Tag individual calls with document, chunk, step, etc. |
| Operation chain | `client.glacis_operation()` | Group calls with shared `operation_id` + auto-incrementing sequence |
| Revision | `op.supersedes(receipt.id)` | Link a replacement to the attestation it replaces |

### API Pattern

```python
from glacis.integrations.openai import attested_openai, get_last_receipt

client = attested_openai(
    offline=True,
    signing_seed=seed,
    metadata={"pipeline": "qa-gen"},  # static, on every call
)

with client.glacis_operation() as op:
    # Step 0: Generate
    with client.glacis_tags({"doc": "Eliquis.pdf", "step": "generate"}):
        gen = client.chat.completions.create(...)
        gen_receipt = get_last_receipt()

    # Step 1: Validate
    with client.glacis_tags({"doc": "Eliquis.pdf", "step": "validate"}):
        val = client.chat.completions.create(...)

    # Step 2: Revise (supersedes generate)
    op.supersedes(gen_receipt.id)
    with client.glacis_tags({"doc": "Eliquis.pdf", "step": "revise"}):
        rev = client.chat.completions.create(...)
```

### Metadata Merge Order

```
provider defaults (provider, model)
  → static metadata (from attested_openai)
    → per-call tags (from glacis_tags)
```

Per-call tags override static tags for overlapping keys. `provider` and `model` are reserved.

### Async Safety

All context managers use `contextvars.ContextVar`, so they are safe for concurrent async calls (e.g., `asyncio.gather` with `acompletion()`). Each coroutine gets its own isolated metadata, operation state, and `get_last_receipt()`.

### See Also

- [Custom Controls Demo](./custom_control_demo.ipynb) — building and wiring custom validation controls
- [Offline Attestation Demo](./offline_attestation_demo.ipynb) — basic attestation and verification